> **Статус миграции:** это пока действующий объединённый источник, а не окончательно разделённый ноутбук.
> 
> Предусмотренное разделение: `30.04`, `31.01–31.03` и `32.01–32.03`. Полная копия происхождения: `archive/legacy/31.90_Объединённый_анализ_чувствительности.ipynb`. Новые каркасы не считаются реализованными до переноса и сравнения вычислений.


# Анализ чувствительности импеданса и обоснование геометрии электродной сборки

**Модель.** Двуслойная среда: верхний слой — мягкие ткани (удельное сопротивление **ρ₁**, толщина **h**, определяемая кожно-жировым слоем), нижний полупространственный слой — лёгкое (**ρ₂**, меняется при дыхании). Тетраполярная сборка: токовые электроды на ±a, потенциальные на ±b; размер сборки **L = 2a**, форма **β = b/a**.

**Что делает notebook.**
- **§1–2.** Считает чувствительность измеряемого импеданса **Z ко всем параметрам** (ρ₁, ρ₂, h, a, b) — и абсолютную (производные), и относительную (эластичности), с интерпретацией.
- **§3.** Объясняет, **откуда берётся усиление ошибки** «X% неточности h → Y% ошибки ρ₂», при каких условиях, и **как оно зависит от геометрии** (размер L vs отношение b/a vs толщина h). Здесь же — что такое **FoM (сигнал/помеха)**.
- **§4–6.** Прикладные задачи: сигнал дыхания, минимальный размер сборки Lₘᵢₙ(h), обоснование набора баз (матрица Фишера).

Все формулы выведены в `04_Чувствительность_и_дизайн_решётки.md`; здесь они реализованы и прокомментированы.

## §0. Методика анализа чувствительности и словарь терминов

Раздел определяет решаемую задачу, метод, обоснование выбора и все термины, используемые далее. Термины не употребляются в последующих разделах раньше, чем определены здесь.

### Решаемая задача
Требуется обосновать геометрию электродных сборок для восстановления удельного сопротивления лёгкого ρ₂ и количественно оценить, как погрешности мешающих параметров (толщины слоя мягких тканей h и удельного сопротивления мягких тканей ρ₁) переносятся на оценку ρ₂. На этой основе определяются минимальный размер сборки, оптимальная форма сборки и оптимальный набор размеров.

### Модель и метод
Прямая задача — двуслойная полубесконечная модель (§0 ноутбука [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb); вывод — [03](30.01_Прямая_двуслойная_модель_боковой_сборки.md)): слой мягких тканей толщины h с удельным сопротивлением ρ₁ над лёгким-полупространством с сопротивлением ρ₂; измерение тетраполярным методом электродной сборкой, форма сборки β = b/a. Метод — аналитические производные импеданса по параметрам (сверяются с численными), безразмерные чувствительности (эластичности) и матрица Фишера. Полный вывод — в [04](31.00_Теоретическая_постановка_чувствительности.md).

### Словарь терминов
- **Электродная сборка** — совокупность четырёх электродов для одного тетраполярного измерения; размер L = 2a (токовая база), форма β = b/a (отношение потенциальной полубазы к токовой).
- **Рабочая точка** — набор значений (L, β, h, ρ₁, ρ₂), в котором вычисляются локальные чувствительности.
- **Эластичность** (относительная чувствительность) — безразмерная величина, показывающая, на сколько процентов изменится импеданс при изменении параметра на один процент:

$$S_p=\frac{\partial \ln Z}{\partial \ln p}=\frac{p}{Z}\,\frac{\partial Z}{\partial p} \tag{5.1}$$

где S_p — эластичность импеданса по параметру p, безразмерная; Z — импеданс, Ом; p — параметр (ρ₁, ρ₂ или h). Эластичность по ρ₂ есть **полезный сигнал** (зависимость от дыхания), по h — **главная помеха**.
- **Показатель качества сигнал/помеха** (FoM, figure of merit — показатель качества) — отношение полезного сигнала к главной помехе:

$$\mathrm{FoM}=\frac{S_{\rho_2}}{|S_h|} \tag{5.2}$$

где FoM — показатель качества, безразмерный. Чем он больше, тем слабее погрешность толщины влияет на оценку ρ₂.
- **Коэффициент усиления ошибки** — во сколько раз относительная погрешность толщины переносится в относительную погрешность ρ₂:

$$K_h=-\frac{S_h}{S_{\rho_2}}=\frac{1}{\mathrm{FoM}} \tag{5.3}$$

где K_h — коэффициент усиления ошибки, безразмерный.
- **Матрица Фишера** — мера информации набора измерений о параметрах; по ней вычисляется нижняя граница ковариации оценок:

$$\mathbf M=J^{\top}\boldsymbol\Sigma^{-1}J,\qquad \mathrm{Cov}(\hat{\boldsymbol\theta})\succeq \mathbf M^{-1} \tag{5.4}$$

где M — матрица Фишера; J — матрица производных импеданса по параметрам (якобиан); Σ — ковариация шума измерений. Из неё: **стандартное отклонение оценки** std(ρ₂) = √(M⁻¹)_{ρ₂ρ₂}; **корреляция помехи и сигнала** corr(h, ρ₂); **число обусловленности** cond(M).
- **c-оптимальность** — выбор геометрии, минимизирующий дисперсию целевой оценки ρ₂: min (M⁻¹)_{ρ₂ρ₂}.
- **Минимальный размер сборки** L_min(h) — наименьший размер, при котором дыхание ещё различимо при толщине h; растёт линейно с толщиной, L_min ∝ h.
- **Индексы Соболя** — доли дисперсии импеданса, вносимые каждым параметром во всём диапазоне значений (глобальная чувствительность, в отличие от локальной эластичности в рабочей точке).
- **Теорема взаимности** — свойство тетраполярного измерения: перестановка ролей токовой и потенциальной пар при фиксированных позициях электродов не меняет импеданс; служит проверкой корректности модели.

### Как оценивать результаты
Геометрия тем лучше для восстановления ρ₂, чем больше показатель качества (5.2) и меньше стандартное отклонение оценки std(ρ₂) и число обусловленности матрицы Фишера. Набор размеров должен покрывать минимальный размер сборки для всего ожидаемого диапазона толщины.

**Входные данные.** Рабочая точка анализа: размер сборки L = 140 мм, форма β = b/a = 0.5, толщина слоя мягких тканей h = 20 мм, удельные сопротивления ρ₁ = 5, ρ₂ = 20 Ом·м.

**Допущения.** Рабочая точка выбрана как типичная для эксперимента; локальные чувствительности зависят от неё, поэтому она фиксируется явно и указывается при каждом результате.

In [ ]:
# @title Импорты и конфигурация
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ============ ФИЗИЧЕСКИЕ ПАРАМЕТРЫ (редактируйте под свои данные) ============
RHO1     = 5.0     # удельное сопротивление мягких тканей,  Ом·м
RHO2_EX  = 15.0    # лёгкое на ВЫДОХЕ (меньше воздуха -> ниже сопротивление), Ом·м
RHO2_IN  = 25.0    # лёгкое на ВДОХЕ (больше воздуха -> выше сопротивление),  Ом·м

# ============ РАБОЧАЯ ТОЧКА для локальных оценок (§2, §3) ============
#   именно при этих значениях считаются таблица чувствительности и усиление ошибки
L_OP     = 0.140   # размер сборки L = 2a, м  (140 мм — текущая макс. база)
BETA_OP  = 0.50    # форма сборки b/a
H_OP     = 0.020   # толщина мягких тканей, м  (20 мм)
RHO2_OP  = 20.0    # рабочее сопротивление лёгкого, Ом·м

# ============ ДИАПАЗОНЫ СКАНИРОВАНИЯ (§3–§6) ============
H_MIN, H_MAX, NH = 0.005, 0.045, 41     # толщина h, м  (5..45 мм)
L_MIN, L_MAX, NL = 0.040, 0.300, 81     # размер L, м   (40..300 мм)

# ============ ЧИСЛЕННЫЕ / ПРИБОРНЫЕ ============
NTERMS   = 400     # число членов ряда зеркальных изображений
SIGMA_Z  = 0.02    # разрешение измерителя импеданса, Ом — ТОЛЬКО для критерия I в §5
# Масштаб фактического расхождения модели и данных, Ом. Не приборный шум: среднеквадратичный
# остаток совместной подгонки (09 §3) равен 7.40 Ом у Ника и 15.10 Ом у Георгия при уровне
# базового импеданса 75 и 155 Ом. В §7 берётся представительное значение между ними, в §8 —
# фактическое значение по каждому испытуемому. Обоснование и оговорки — в тексте перед §7.
C_SNR    = 3.0     # требуемое отношение сигнал/шум
SIGMA_FIT = 10.0   # представительный масштаб невязки модель-данные, Ом (см. выше)

print('Конфигурация задана.')
print('Рабочая точка: L=%.0f мм, b/a=%.2f, h=%.0f мм, rho1=%.1f, rho2=%.1f Ом·м'
      % (L_OP*1000, BETA_OP, H_OP*1000, RHO1, RHO2_OP))

**Анализ результатов.** Ячейка задаёт конфигурацию и выводит рабочую точку.

**Результаты и умозаключения.** Рабочая точка — общий вход для всех разделов, вычисляющих локальные чувствительности.

## §1. Прямая модель и её производные

Определение импеданса двуслойной модели (5.1 — см. §0) и его аналитических производных по всем параметрам; производные нужны для эластичностей и матрицы Фишера.

**Входные данные.** Параметры прямой задачи и рабочая точка из §0.

**Допущения.** Импеданс двуслойной модели и его производные по ρ₁, ρ₂, h, a, b заданы в замкнутой форме (вывод — [04](31.00_Теоретическая_постановка_чувствительности.md)); это избавляет от конечных разностей и даёт точные эластичности (5.1).

In [ ]:
# @title Модель Z и аналитические производные по всем параметрам
def _terms(rho1, rho2, h, a, b, N=NTERMS):
    # Общие промежуточные величины ряда зеркальных изображений
    k  = (rho2 - rho1) / (rho1 + rho2)        # контраст слоёв
    d1, d2 = a - b, a + b                      # расстояния между электродными парами
    i  = np.arange(1, N + 1)                   # номера зеркальных изображений
    G1 = 1.0 / np.sqrt(d1**2 + (2*i*h)**2)     # G_i(d1)
    G2 = 1.0 / np.sqrt(d2**2 + (2*i*h)**2)     # G_i(d2)
    return k, d1, d2, i, G1, G2, (G1 - G2)

def Z_model(rho1, rho2, h, a, b, N=NTERMS):
    # Импеданс, Ом (формула (2))
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    return rho1/np.pi*(1/d1 - 1/d2) + 2*rho1/np.pi*np.sum(k**i * dG)

# --- производные по ФИЗИЧЕСКИМ параметрам ---
def dZ_drho2(rho1, rho2, h, a, b, N=NTERMS):
    # ∂Z/∂ρ2 — реакция на сопротивление лёгкого (ДЫХАНИЕ)
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    T = np.sum(i * k**(i-1) * dG)                       # вспомогательный ряд T
    return 4*rho1**2 / (np.pi*(rho1+rho2)**2) * T

def dZ_drho1(rho1, rho2, h, a, b, N=NTERMS):
    # ∂Z/∂ρ1 — реакция на сопротивление мягких тканей
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    T = np.sum(i * k**(i-1) * dG)
    return Z_model(rho1, rho2, h, a, b, N)/rho1 - 4*rho1*rho2/(np.pi*(rho1+rho2)**2)*T

def dZ_dh(rho1, rho2, h, a, b, N=NTERMS):
    # ∂Z/∂h — реакция на толщину мягких тканей (кожно-жировой слой = ПОМЕХА)
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    U = np.sum(i**2 * k**i * (G1**3 - G2**3))           # вспомогательный ряд U
    return -8*rho1*h/np.pi * U

# --- производные по ГЕОМЕТРИИ (для матрицы Фишера и оценки точности установки) ---
def dZ_da(rho1, rho2, h, a, b, N=NTERMS):
    # ∂Z/∂a — реакция на полубазу токовых электродов
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    return rho1/np.pi*(1/d2**2 - 1/d1**2) + 2*rho1/np.pi*np.sum(k**i*(d2*G2**3 - d1*G1**3))

def dZ_db(rho1, rho2, h, a, b, N=NTERMS):
    # ∂Z/∂b — реакция на полубазу потенциальных электродов
    k, d1, d2, i, G1, G2, dG = _terms(rho1, rho2, h, a, b, N)
    return rho1/np.pi*(1/d1**2 + 1/d2**2) + 2*rho1/np.pi*np.sum(k**i*(d1*G1**3 + d2*G2**3))

# --- сервис ---
def rho_apparent(Z, a, b):
    # Кажущееся удельное сопротивление: обращение однослойной формулы
    return Z * np.pi * (a**2 - b**2) / (2*b)

def ab_of(L, beta=BETA_OP):
    # Геометрия (a, b) по размеру L=2a и отношению beta=b/a
    a = L/2.0
    return a, beta*a

print('Модель и производные определены.')

**Анализ результатов.** Функции модели и производных определены.

**Результаты и умозаключения.** Определённые производные — вход для §1.1 (самоконтроль), §2 (эластичности) и §6 (матрица Фишера).

### §1.1. Самоконтроль кода: производные и инварианты

**Входные данные.** Аналитические производные из §1 и их численные оценки конечными разностями в рабочей точке.

**Допущения.** Чтобы убедиться в корректности аналитических производных, они сравниваются с численными; дополнительно проверяются инварианты однородности импеданса по сопротивлениям и по геометрии.

In [ ]:
# @title Проверка производных и инвариантов
a, b = ab_of(L_OP, BETA_OP)
args = (RHO1, RHO2_OP, H_OP, a, b)

e = 1e-7  # шаг центральной разности
checks = {
 '∂Z/∂ρ2': (dZ_drho2(*args), (Z_model(RHO1,RHO2_OP+e,H_OP,a,b)-Z_model(RHO1,RHO2_OP-e,H_OP,a,b))/(2*e)),
 '∂Z/∂ρ1': (dZ_drho1(*args), (Z_model(RHO1+e,RHO2_OP,H_OP,a,b)-Z_model(RHO1-e,RHO2_OP,H_OP,a,b))/(2*e)),
 '∂Z/∂h ': (dZ_dh(*args),    (Z_model(RHO1,RHO2_OP,H_OP+1e-9,a,b)-Z_model(RHO1,RHO2_OP,H_OP-1e-9,a,b))/(2e-9)),
 '∂Z/∂a ': (dZ_da(*args),    (Z_model(RHO1,RHO2_OP,H_OP,a+e,b)-Z_model(RHO1,RHO2_OP,H_OP,a-e,b))/(2*e)),
 '∂Z/∂b ': (dZ_db(*args),    (Z_model(RHO1,RHO2_OP,H_OP,a,b+e)-Z_model(RHO1,RHO2_OP,H_OP,a,b-e))/(2*e)),
}
print('производная | аналитически   | численно       | отн.расхождение')
for n,(an,nu) in checks.items():
    print('  %s   | %+.6e | %+.6e | %.1e' % (n, an, nu, abs(an-nu)/abs(nu)))

Z = Z_model(*args)
inv1 = (RHO1*dZ_drho1(*args) + RHO2_OP*dZ_drho2(*args)) / Z
inv2 = (a*dZ_da(*args) + b*dZ_db(*args) + H_OP*dZ_dh(*args)) / Z
print('\nИнвариант 1: (ρ1·Zρ1+ρ2·Zρ2)/Z = %.6f  (ожидаем +1)' % inv1)
print('Инвариант 2: (a·Za+b·Zb+h·Zh)/Z = %.6f  (ожидаем −1)' % inv2)

**Анализ результатов.** Аналитические и численные производные совпадают с относительным расхождением 10⁻⁸…10⁻¹³. Инвариант по сопротивлениям (ρ₁·∂Z/∂ρ₁ + ρ₂·∂Z/∂ρ₂)/Z = +1 и инвариант по геометрии (a·∂Z/∂a + b·∂Z/∂b + h·∂Z/∂h)/Z = −1 выполнены точно.

**Результаты и умозаключения.** Производные подтверждены; их можно использовать в эластичностях и матрице Фишера без опасения ошибок дифференцирования.

### §1.2. Самоконтроль: теорема взаимности

**Входные данные.** Импеданс, вычисленный при прямом и переставленном назначении ролей токовой и потенциальной пар электродов.

**Допущения.** Теорема взаимности (§0) требует равенства импеданса при перестановке ролей пар; это проверка корректности модели. Отдельно показывается, что подстановка b > a в замкнутую формулу — не взаимность, а выход за область применимости (формула выведена для a > b).

In [ ]:
# @title Проверка теоремы взаимности
# Передаточный импеданс по ЯВНЫМ координатам 4 электродов на линии (двуслойная среда)
def Z_tetra_coords(c_plus, c_minus, v_plus, v_minus, rho1, rho2, h, N=NTERMS):
    k = (rho2 - rho1) / (rho1 + rho2)
    i = np.arange(1, N + 1)
    def Kern(d):                      # ядро: полупространство (1/d) + ряд изображений границы
        return 1/d + 2*np.sum(k**i / np.sqrt(d**2 + (2*i*h)**2))
    f = lambda v, c: Kern(abs(v - c))
    return rho1/(2*np.pi)*(f(v_plus,c_plus) - f(v_plus,c_minus) - f(v_minus,c_plus) + f(v_minus,c_minus))

a, b = ab_of(L_OP, BETA_OP)          # a — токовая (внешняя) полубаза, b — потенциальная (внутренняя)
Z_AB = Z_tetra_coords(+a, -a, +b, -b, RHO1, RHO2_OP, H_OP)   # ток@±a, потенциал@±b
Z_BA = Z_tetra_coords(+b, -b, +a, -a, RHO1, RHO2_OP, H_OP)   # РОЛИ переставлены: ток@±b, потенциал@±a

print('Взаимность (перестановка ролей пар при фиксированных позициях):')
print('  ток@±a, потенциал@±b :  Z = %.6f Ом' % Z_AB)
print('  ток@±b, потенциал@±a :  Z = %.6f Ом' % Z_BA)
print('  совпадают: %s  -> взаимность выполнена' % np.isclose(Z_AB, Z_BA))
print('  Z_model(a,b)         = %.6f Ом  (та же величина)' % Z_model(RHO1, RHO2_OP, H_OP, a, b))
print()
print('Литеральная подстановка b>a в замкнутую формулу — НЕ взаимность, а выход за область применимости:')
print('  Z_model(a=b_inner, b=a_outer) = %.4f Ом  (неверно: формула выведена для a>b)'
      % Z_model(RHO1, RHO2_OP, H_OP, b, a))

**Анализ результатов.** При перестановке ролей пар импеданс совпадает (55.44 Ом в обоих случаях) — взаимность выполнена. Подстановка b > a даёт отрицательный импеданс (−35.5 Ом), что подтверждает: формула применима только при a > b.

**Результаты и умозаключения.** Модель корректна в области a > b; вне её результаты недостоверны — это ограничение учитывается при переборе форм β в §8.

## §2. Чувствительность импеданса ко всем параметрам в рабочей точке

**Входные данные.** Производные импеданса и рабочая точка из §0.

**Допущения.** Чтобы сопоставить влияние всех параметров в единой безразмерной шкале, вычисляются эластичности (5.1) по ρ₁, ρ₂, h, a, b.

In [ ]:
# @title Таблица и диаграмма чувствительности ко всем параметрам
a, b = ab_of(L_OP, BETA_OP)
Z = Z_model(RHO1, RHO2_OP, H_OP, a, b)

# (имя, значение параметра, производная, краткий смысл)
rows = [
 ('ρ1', RHO1,    dZ_drho1(RHO1,RHO2_OP,H_OP,a,b), 'сопр. мягких тканей'),
 ('ρ2', RHO2_OP, dZ_drho2(RHO1,RHO2_OP,H_OP,a,b), 'сопр. лёгкого = ДЫХАНИЕ (сигнал)'),
 ('h',  H_OP,    dZ_dh   (RHO1,RHO2_OP,H_OP,a,b), 'толщина жира = ПОМЕХА'),
 ('a',  a,       dZ_da   (RHO1,RHO2_OP,H_OP,a,b), 'токовая полубаза (геометрия)'),
 ('b',  b,       dZ_db   (RHO1,RHO2_OP,H_OP,a,b), 'потенц. полубаза (геометрия)'),
]

print('параметр |   ∂Z/∂p      | эластичность S_p | трактовка: +1%% параметра -> ... %% Z')
print('-'*92)
names, S = [], []
for nm, p, dZ, meaning in rows:
    Sp = p/Z*dZ                          # эластичность
    names.append(nm); S.append(Sp)
    print('  %-3s    | %+.4e | %+8.3f       | %+6.2f%%   (%s)' % (nm, dZ, Sp, Sp, meaning))

# диаграмма эластичностей
plt.figure(figsize=(8,4.5))
colors = ['C0' if s>=0 else 'C3' for s in S]
plt.barh(names[::-1], S[::-1], color=colors[::-1])
plt.axvline(0, color='k', lw=0.8)
plt.xlabel('эластичность $S_p=\\partial\\ln Z/\\partial\\ln p$  (% изменения Z на 1% параметра)')
plt.title('Чувствительность Z ко всем параметрам (L=%.0fмм, b/a=%.2f, h=%.0fмм)'%(L_OP*1000,BETA_OP,H_OP*1000))
plt.grid(True, axis='x'); plt.tight_layout(); plt.show()

**Анализ результатов.** В рабочей точке эластичность по ρ₂ (сигнал дыхания) равна +0.33, по ρ₁ +0.67, по h (помеха) −0.54; геометрические эластичности по a и b по модулю больше единицы (−1.84 и +1.38), то есть импеданс наиболее чувствителен к точности установки электродов.

**Результаты и умозаключения.** Полезный сигнал (эластичность по ρ₂) меньше помехи (эластичности по h и ρ₁); отсюда следует необходимость дизайна геометрии, повышающего сигнал относительно помехи (§3, §6, §8). Высокая чувствительность к a, b задаёт требования к точности установки сборок.

### §2.1. Как читать результат

- **ρ₁ и ρ₂** — физика среды. Их эластичности в сумме дают 1 ($S_{\rho_1}+S_{\rho_2}=1$). В рабочей точке $S_{\rho_1}\approx0.67$, $S_{\rho_2}\approx0.33$ — то есть **мягкие ткани влияют на Z сильнее, чем лёгкое**: дыхательный сигнал ($\rho_2$) — лишь треть отклика. Это и есть корень проблемы: полезный сигнал слабее фона.
- **h** — отрицательная эластичность ($\approx-0.54$): толще жир ⟹ меньше Z. По модулю **сравнима с $S_{\rho_1}$ и больше $S_{\rho_2}$** — поэтому неопределённость толщины так опасна (§3).
- **a, b** — самые большие по модулю эластичности ($S_a\approx-1.8$, $S_b\approx+1.4$): Z очень чувствителен к **точности установки электродов**. Сдвиг электрода на 1 % уже меняет Z на 1–2 %. Вывод: позиционирование сборки должно быть точным (или учитываться в модели).

## §3. Перенос погрешности толщины на ρ₂ и показатель качества

**Входные данные.** Эластичности по ρ₂ и h в рабочей точке из §2.

**Допущения.** Чтобы выразить, как погрешность толщины переносится на ρ₂, вычисляются показатель качества (5.2) и коэффициент усиления ошибки (5.3).

In [ ]:
# @title Усиление ошибки в рабочей точке — с явными условиями
def Sr2_Sh(rho1, rho2, h, a, b):
    Z = Z_model(rho1, rho2, h, a, b)
    return rho2/Z*dZ_drho2(rho1,rho2,h,a,b), h/Z*dZ_dh(rho1,rho2,h,a,b)

def K_h(rho1, rho2, h, a, b):   # усиление ошибки  d ln rho2 / d ln h
    Sr2, Sh = Sr2_Sh(rho1, rho2, h, a, b)
    return -Sh/Sr2

a, b = ab_of(L_OP, BETA_OP)
Sr2, Sh = Sr2_Sh(RHO1, RHO2_OP, H_OP, a, b)
kh  = -Sh/Sr2
fom = Sr2/abs(Sh)
print('УСЛОВИЯ (рабочая точка): L=%.0f мм, b/a=%.2f, h=%.0f мм, rho1=%.1f, rho2=%.1f'
      % (L_OP*1000, BETA_OP, H_OP*1000, RHO1, RHO2_OP))
print('-'*64)
print('  сигнал  S_rho2 = %+.3f' % Sr2)
print('  помеха  S_h    = %+.3f' % Sh)
print('  FoM = S_rho2/|S_h| = %.3f' % fom)
print('  усиление ошибки K_h = -S_h/S_rho2 = %.2f' % kh)
print('  =>  10%% ошибки в h  ->  %.0f%% ошибки в rho2' % (10*abs(kh)))
print('  =>   5%% ошибки в h  ->  %.0f%% ошибки в rho2' % (5*abs(kh)))
print('\nИменно отсюда «10%% -> 17%%»: это значение K_h ИМЕННО в этой точке и зависит от геометрии (см. ниже).')

**Анализ результатов.** В рабочей точке показатель качества FoM = 0.60, коэффициент усиления ошибки K_h = 1.66; следовательно, 10 % погрешности толщины дают около 17 % погрешности ρ₂.

**Результаты и умозаключения.** Соотношение «10 % → 17 %» — прямое следствие геометрии рабочей точки. Так как K_h > 1, толщину нужно знать точнее, чем требуемая точность ρ₂; это обосновывает измерение толщины по томографии (§0 ноутбука 09).

### §3.1. От чего зависит усиление ошибки: размер сборки, форма и толщина

**Входные данные.** Коэффициент усиления ошибки (5.3) как функция размера сборки L, формы β и толщины h.

**Допущения.** Чтобы понять, чем управляется усиление ошибки, K_h строится в зависимости от каждого из трёх факторов по отдельности.

In [ ]:
# @title K_h как функция L, b/a и h — три графика
Ls   = np.linspace(0.05, 0.30, 60)
betas= np.linspace(0.15, 0.85, 60)
hs1  = np.linspace(0.006, 0.045, 60)
a_fix = L_OP/2

Kh_L    = np.array([K_h(RHO1,RHO2_OP,H_OP,*ab_of(L,BETA_OP)) for L in Ls])
Kh_beta = np.array([K_h(RHO1,RHO2_OP,H_OP, a_fix, be*a_fix) for be in betas])
Kh_h    = np.array([K_h(RHO1,RHO2_OP,h,   *ab_of(L_OP,BETA_OP)) for h in hs1])

fig, ax = plt.subplots(1, 3, figsize=(18,5))
ax[0].plot(Ls*1000, np.abs(Kh_L)); ax[0].axvline(L_OP*1000, color='k', ls='--', lw=1)
ax[0].set_xlabel('размер L, мм'); ax[0].set_ylabel('|K_h|  (усиление ошибки h)')
ax[0].set_title('vs РАЗМЕР  (b/a=%.2f, h=%.0fмм)'%(BETA_OP,H_OP*1000)); ax[0].grid(True)

ax[1].plot(betas, np.abs(Kh_beta)); ax[1].axvline(BETA_OP, color='k', ls='--', lw=1)
ax[1].set_xlabel('отношение b/a'); ax[1].set_ylabel('|K_h|')
ax[1].set_title('vs ОТНОШЕНИЕ b/a  (L=%.0fмм, h=%.0fмм)'%(L_OP*1000,H_OP*1000)); ax[1].grid(True)

ax[2].plot(hs1*1000, np.abs(Kh_h)); ax[2].axvline(H_OP*1000, color='k', ls='--', lw=1)
ax[2].set_xlabel('толщина h, мм'); ax[2].set_ylabel('|K_h|')
ax[2].set_title('vs ТОЛЩИНА h  (L=%.0fмм, b/a=%.2f)'%(L_OP*1000,BETA_OP)); ax[2].grid(True)
plt.tight_layout(); plt.show()

print('Размах |K_h|:  по L: %.2f..%.2f | по b/a: %.2f..%.2f | по h: %.2f..%.2f'
      % (np.abs(Kh_L).min(),np.abs(Kh_L).max(),
         np.abs(Kh_beta).min(),np.abs(Kh_beta).max(),
         np.abs(Kh_h).min(),np.abs(Kh_h).max()))

**Анализ результатов.** Размах |K_h|: по размеру сборки 0.74…3.58, по форме 1.44…2.04, по толщине 0.44…3.17. Наибольшее влияние оказывают размер сборки и толщина.

**Результаты и умозаключения.** Усиление ошибки уменьшается с ростом размера сборки и уменьшением толщины; это направление и используется при оптимизации геометрии (§8).

**Входные данные.** Коэффициент усиления ошибки (5.3) на сетке размеров сборки L и форм β при фиксированной толщине h = 20 мм.

**Допущения.** Чтобы найти лучшую комбинацию размера и формы, строится карта |K_h|(L, β).

In [ ]:
# @title Карта |K_h|(L, b/a) при фиксированной толщине — лучшая комбинация
Lg   = np.linspace(0.05, 0.30, 50)
bg   = np.linspace(0.15, 0.85, 50)
LG, BG = np.meshgrid(Lg, bg)
KhMap = np.zeros_like(LG)
for ii in range(LG.shape[0]):
    for jj in range(LG.shape[1]):
        a = LG[ii,jj]/2
        KhMap[ii,jj] = abs(K_h(RHO1, RHO2_OP, H_OP, a, BG[ii,jj]*a))

plt.figure(figsize=(9,6))
im = plt.imshow(KhMap, origin='lower', aspect='auto',
                extent=[Lg[0]*1000, Lg[-1]*1000, bg[0], bg[-1]], cmap='RdYlGn_r',
                vmin=KhMap.min(), vmax=min(KhMap.max(), 4))
plt.colorbar(im, label='|K_h| (меньше = лучше)')
cs = plt.contour(LG*1000, BG, KhMap, levels=[0.5,1.0,1.5,2.0,3.0], colors='k', linewidths=0.7)
plt.clabel(cs, fmt='%.1f')
i0,j0 = np.unravel_index(np.argmin(KhMap), KhMap.shape)
plt.plot(Lg[j0]*1000, bg[i0], 'b*', ms=18, label='минимум |K_h|=%.2f'%KhMap.min())
plt.plot(L_OP*1000, BETA_OP, 'ko', ms=10, label='текущая точка')
plt.xlabel('размер L, мм'); plt.ylabel('отношение b/a')
plt.title('Усиление ошибки h при h=%.0f мм: где обратная задача устойчивее'%(H_OP*1000))
plt.legend(); plt.tight_layout(); plt.show()
print('Лучшая комбинация при h=%.0fмм: L=%.0f мм, b/a=%.2f -> |K_h|=%.2f'
      % (H_OP*1000, Lg[j0]*1000, bg[i0], KhMap.min()))

**Анализ результатов.** Минимум |K_h| = 0.59 достигается при большом размере сборки (L = 300 мм) и малой форме (β = 0.15). Малое β повышает сигнал, но снижает измеряемое напряжение; крупный размер ограничен плоскостностью модели.

**Результаты и умозаключения.** Направление к меньшему усилению ошибки — крупные размеры и малые β; практический выбор ограничен верхней границей размера (плоскостность) и минимально допустимым напряжением, что учитывается в §8.

### §3.2. Выводы по усилению ошибки

- **Сильнее всего — от размера L.** Увеличение базы резко снижает $|K_h|$ (крупная сборка глубже «видит» лёгкое ⟹ дыхательный сигнал $S_{\rho_2}$ растёт ⟹ влияние неопределённости $h$ относительно слабеет). Это **главный рычаг** борьбы с неопределённостью жирового слоя.
- **Слабее — от отношения b/a.** Меньшее $b/a$ немного улучшает $|K_h|$, но эффект вторичен по сравнению с размером.
- **Толщина h — это сама помеха, а не рычаг.** Чем толще кожно-жировой слой, тем больше $|K_h|$: у «полных» пациентов та же относительная неопределённость $h$ сильнее портит $\rho_2$. Поэтому для них особенно важны крупные базы и/или **независимое измерение $h$** (УЗИ/МРТ).
- **Лучшая геометрия:** максимально возможный $L$ при умеренно малом $b/a$ (см. синюю звезду на карте).

## §4. Дыхательный сигнал $Z(\rho_2)$ в зависимости от размера сборки и толщины

**Входные данные.** Дыхательный размах импеданса как функция размера сборки L и толщины h (при заданном изменении ρ₂ между вдохом и выдохом).

**Допущения.** Чтобы показать область информативности, строятся карты дыхательного сигнала. Абсолютный размах убывает с размером сборки (импеданс масштабируется как ρ₁/a), а относительный размах и эластичность по ρ₂ растут с размером.

In [ ]:
# @title Карты сигнала дыхания
Ls = np.linspace(L_MIN, L_MAX, NL)
hs = np.linspace(H_MIN, H_MAX, NH)
LL, HH = np.meshgrid(Ls, hs)
Sr2_map, dZ_map = np.zeros_like(LL), np.zeros_like(LL)
for ii in range(LL.shape[0]):
    for jj in range(LL.shape[1]):
        a, b = ab_of(LL[ii,jj], BETA_OP)
        Sr2_map[ii,jj] = Sr2_Sh(RHO1, RHO2_OP, HH[ii,jj], a, b)[0]
        dZ_map[ii,jj]  = (Z_model(RHO1,RHO2_IN,HH[ii,jj],a,b)
                          - Z_model(RHO1,RHO2_EX,HH[ii,jj],a,b))

fig, ax = plt.subplots(1, 2, figsize=(14,5))
ext = [Ls[0]*1000, Ls[-1]*1000, hs[0]*1000, hs[-1]*1000]
for k,(M,ttl,lab) in enumerate([(Sr2_map,'$S_{\\rho_2}$ — эластичность к дыханию (сигнал)','$S_{\\rho_2}$'),
                                 (dZ_map,'$\\Delta Z_{дых}$ — абсолютный размах, Ом','Ом')]):
    im = ax[k].imshow(M, origin='lower', aspect='auto', extent=ext, cmap='viridis')
    ax[k].set_title(ttl); ax[k].set_xlabel('L, мм'); ax[k].set_ylabel('h, мм')
    ax[k].axvline(140, color='w', ls='--', lw=1)
    fig.colorbar(im, ax=ax[k], label=lab)
plt.tight_layout(); plt.show()
print('Белый пунктир — текущая макс. база 140 мм. Сигнал растёт вправо (база) и вниз (тоньше жир).')

**Анализ результатов.** Сигнал растёт при увеличении размера сборки и уменьшении толщины; текущая максимальная база 140 мм отмечена на карте.

**Результаты и умозаключения.** Крупные сборки информативнее для ρ₂ по относительному сигналу; это согласуется с выводом §3 и обосновывает расширение набора размеров вверх (§6, §8).

## §5. Минимальный размер сборки $L_{\min}(h)$

**Входные данные.** Модель, эластичности и три критерия минимального размера сборки (по измеримости сигнала, по глубине проникновения, по идентифицируемости).

**Допущения.** Чтобы определить наименьший пригодный размер сборки при данной толщине, вычисляется L_min(h) по трём критериям; основной — по идентифицируемости (порог показателя качества).

In [ ]:
# @title Lmin(h) по трём критериям
F_STAR, S_STAR, FOM_STAR = 0.10, 0.30, 1.0
Lscan = np.linspace(L_MIN, L_MAX, 400)

def Lmin_at(h):
    out = {}
    dZ = np.array([Z_model(RHO1,RHO2_IN,h,*ab_of(L)) - Z_model(RHO1,RHO2_EX,h,*ab_of(L)) for L in Lscan])
    out['I (шум)'] = next((L for L,v in zip(Lscan,dZ) if v >= C_SNR*SIGMA_Z), np.nan)
    F  = np.array([(rho_apparent(Z_model(RHO1,RHO2_OP,h,*ab_of(L)),*ab_of(L))-RHO1)/(RHO2_OP-RHO1) for L in Lscan])
    out['II (проникн.)'] = next((L for L,v in zip(Lscan,F) if v >= F_STAR), np.nan)
    SF = np.array([Sr2_Sh(RHO1,RHO2_OP,h,*ab_of(L)) for L in Lscan])
    Sr = SF[:,0]; fom = SF[:,0]/np.abs(SF[:,1])
    out['III (идентиф.)'] = next((L for L,s,f in zip(Lscan,Sr,fom) if s>=S_STAR and f>=FOM_STAR), np.nan)
    return out

curves = {c: [] for c in ['I (шум)','II (проникн.)','III (идентиф.)']}
for h in hs:
    r = Lmin_at(h)
    for c in curves: curves[c].append(r[c])

plt.figure(figsize=(8,6))
for c,st in zip(curves, ['o-','s-','^-']):
    plt.plot(hs*1000, np.array(curves[c])*1000, st, ms=4, label=c)
yIII = np.array(curves['III (идентиф.)'])*1000; m = np.isfinite(yIII)
p = np.polyfit(hs[m]*1000, yIII[m], 1)
plt.plot(hs*1000, np.polyval(p, hs*1000), 'k--', lw=1, label='лин. подгонка III: %.1f·h+%.0f'%(p[0],p[1]))
plt.axhspan(50,140, color='green', alpha=0.12, label='текущий диапазон 50–140 мм')
plt.xlabel('толщина мягких тканей h, мм'); plt.ylabel('$L_{min}$, мм')
plt.title('Минимальный размер решётки от толщины'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
print('Наклон Lmin(III) ≈ %.1f -> подтверждает линейный закон Lmin ∝ h.' % p[0])
print('Где кривая выходит выше зелёной зоны — текущих баз не хватает (толстый жировой слой).')

**Анализ результатов.** Наклон зависимости L_min от толщины около 11.4, что подтверждает линейный закон L_min ∝ h. При большой толщине текущих размеров сборок не хватает.

**Результаты и умозаключения.** Диапазон размеров должен покрывать L_min(h) для всего ожидаемого разброса толщины; иначе для толстого слоя мягких тканей сигнал недостаточен — это вход для оптимизации набора (§8).

## §6. Обоснование набора размеров сборок — матрица Фишера

**Входные данные.** Матрица Фишера (5.4) для нескольких наборов размеров сборок при равном шуме измерений.

**Допущения.** Чтобы сравнить наборы размеров, для каждого вычисляются стандартное отклонение оценки ρ₂, корреляция толщины и ρ₂, число обусловленности матрицы Фишера.

Масштаб шума σ, входящий в матрицу Фишера, принимается равным фактическому расхождению модели и данных, а не разрешению измерителя. Разрешение измерителя (0.02 Ом) на два-три порядка меньше среднеквадратичного остатка совместной подгонки (7.40 и 15.10 Ом, [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3), поэтому подстановка разрешения давала бы стандартное отклонение оценки, заниженное во столько же раз. Стандартное отклонение оценки пропорционально σ, поэтому сравнение наборов размеров между собой от этого выбора не зависит, а абсолютные значения зависят прямо.

Принятое σ является эффективным: остаток подгонки не сводится к некоррелированному шуму измерения, поскольку содержит воспроизводимую систематику, связанную с размером сборки, величиной 3.9–4.8 % ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §2.2) и ошибку самой двуслойной модели. Для коррелированной составляющей граница Крамера–Рао неприменима, поэтому получаемое стандартное отклонение следует читать как оценку снизу для достижимой неопределённости и как оценку сверху для достижимой точности.

In [ ]:
# @title Сравнение наборов баз
def fisher(L_set, rho1, h, rho2, sigma=SIGMA_Z, beta=BETA_OP):
    J = np.array([[dZ_drho1(rho1,rho2,h,*ab_of(L,beta)),
                   dZ_dh   (rho1,rho2,h,*ab_of(L,beta)),
                   dZ_drho2(rho1,rho2,h,*ab_of(L,beta))] for L in L_set])
    M = J.T @ J / sigma**2
    C = np.linalg.inv(M)
    return dict(std_rho2=np.sqrt(C[2,2]),
                corr=C[1,2]/np.sqrt(C[1,1]*C[2,2]),
                cond=np.linalg.cond(M))

designs = {
 'текущий 50–140 (шаг 10)':     np.arange(50,141,10)/1000,
 'расширенный 50–200 (шаг 15)': np.arange(50,201,15)/1000,
 'расширенный 50–300 (шаг 25)': np.arange(50,301,25)/1000,
 'только крупные 100–200':      np.arange(100,201,20)/1000,
}
print('%-30s | std(rho2) | corr(h,rho2) | cond(M)' % 'набор баз')
print('-'*78)
for name, Ls_ in designs.items():
    r = fisher(Ls_, RHO1, H_OP, RHO2_OP, sigma=SIGMA_FIT)
    print('%-30s |  %7.3f  |    %5.2f     | %.1e' % (name, r['std_rho2'], r['corr'], r['cond']))
print('\nВывод: расширение баз вверх снижает std(rho2) и корреляцию h↔rho2 (разрывает вырождение).')
print('std(rho2) пропорционально sigma; здесь sigma=%.2f Ом — фактическая невязка, не разрешение прибора.' % SIGMA_FIT)

**Анализ результатов.** При масштабе невязки σ = 10 Ом текущий набор 50–140 мм даёт стандартное отклонение оценки ρ₂, равное 39.1 Ом·м, корреляцию толщины и ρ₂ 0.97 и число обусловленности 3.9·10⁸. Расширение вверх до 200 и 300 мм снижает стандартное отклонение до 18.3 и 10.7 Ом·м и уменьшает корреляцию до 0.94 и 0.89. Набор только из крупных сборок 100–200 мм даёт 54.0 Ом·м, то есть хуже текущего: без мелких сборок ρ₁ и h не закрепляются, и вырождение усиливается.

Абсолютные значения следует читать в сопоставлении с самим ρ₂ (порядка 20 Ом·м): стандартное отклонение 39.1 Ом·м означает, что при текущем наборе и фактическом уровне расхождения модели с данными ρ₂ не определяется вовсе. Это согласуется с прямым наблюдением обратной задачи ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3.1), где профиль невязки по ρ₂ не имеет минимума внутри допустимого диапазона.

**Результаты и умозаключения.** Расширение набора вверх снижает неопределённость ρ₂ и разрывает вырождение между толщиной и ρ₂; это обоснование расширения диапазона размеров (§8), ограниченного сверху плоскостностью модели. Отношение стандартных отклонений между наборами не зависит от принятого σ, поэтому вывод о полезности расширения устойчив; абсолютные значения пропорциональны σ и потому улучшатся вместе со снижением расхождения модели и данных — в частности, после устранения систематики отдельных сборок ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §2.2.1).

**Входные данные.** Стандартное отклонение оценки ρ₂ по матрице Фишера (5.4) для наборов размеров при разной толщине h.

**Допущения.** Чтобы проверить пригодность набора во всём диапазоне толщины, стандартное отклонение оценки ρ₂ вычисляется как функция толщины.

In [ ]:
# @title Робастность набора по толщине h
plt.figure(figsize=(8,5))
for name, Ls_ in designs.items():
    std = [fisher(Ls_, RHO1, h, RHO2_OP)['std_rho2'] for h in hs]
    plt.plot(hs*1000, std, label=name)
plt.xlabel('толщина h, мм'); plt.ylabel('std($\\hat\\rho_2$), Ом·м (нижняя граница Крамера–Рао)')
plt.yscale('log'); plt.title('Неопределённость rho2 vs толщина для разных наборов баз')
plt.legend(); plt.grid(True, which='both'); plt.tight_layout(); plt.show()
print('Для больших h текущий набор даёт резко худшую оценку — нужны крупные базы.')

**Анализ результатов.** При большой толщине текущий набор даёт резко худшую оценку ρ₂; нужны крупные размеры сборок.

**Результаты и умозаключения.** Робастный по толщине набор должен включать крупные размеры; это учитывается при оптимизации (§8) и объясняет неопределимость ρ₂ у толстого испытуемого ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3.1).

## §7. Глобальная чувствительность — индексы Соболя (опционально)

**Входные данные.** Модель и физиологические диапазоны параметров для выборки Соболя (при установленной библиотеке SALib).

**Допущения.** Чтобы оценить вклад каждого параметра во всём диапазоне значений, а не только в рабочей точке, вычисляются индексы Соболя (§0).

In [ ]:
# @title Индексы Соболя по размеру L (если установлен SALib)
try:
    from SALib.sample import saltelli
    from SALib.analyze import sobol
    problem = {'num_vars':3, 'names':['rho1','rho2','h'],
               'bounds':[[3.0,7.0],[RHO2_EX,RHO2_IN],[H_MIN,H_MAX]]}
    L_list = np.array([0.06,0.10,0.14,0.18,0.24])
    S1 = {n:[] for n in problem['names']}
    X = saltelli.sample(problem, 256, calc_second_order=False)
    for L in L_list:
        a,b = ab_of(L)
        Y = np.array([Z_model(r1,r2,h,a,b) for r1,r2,h in X])
        Si = sobol.analyze(problem, Y, calc_second_order=False, print_to_console=False)
        for j,n in enumerate(problem['names']): S1[n].append(Si['S1'][j])
    plt.figure(figsize=(8,5))
    lab = {'rho1':'ρ1 (мягкие ткани)','rho2':'ρ2 (дыхание)','h':'h (толщина)'}
    for n in problem['names']:
        plt.plot(L_list*1000, S1[n], 'o-', label=lab[n])
    plt.xlabel('L, мм'); plt.ylabel('индекс Соболя 1-го порядка $S_1$')
    plt.title('Глобальная доля дисперсии Z по параметрам'); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
    print('Доля дисперсии от ρ2 (дыхание) растёт с базой; h остаётся весомой помехой.')
except ImportError:
    print('SALib не установлен. Установите: pip install SALib  (или пропустите эту ячейку).')

**Анализ результатов.** Доля дисперсии импеданса, вносимая ρ₂ (дыханием), растёт с размером сборки, тогда как толщина остаётся весомой помехой во всём диапазоне.

**Результаты и умозаключения.** Глобальный анализ подтверждает локальный вывод §2–§3: крупные размеры информативнее для ρ₂, толщина — главная помеха.

## §8. Оптимизация геометрии сборки $(L,\beta)$ на калиброванных данными параметрах

**Входные данные.** Калиброванная данными рабочая точка (ρ₁, ρ₂, h) из [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) и [10](20.01_DICOM_QC_и_геометрия_тома.ipynb); ограничения на размер сборки (нижнее — минимальный размер, верхнее — плоскостность модели, задано заглушкой L_max = 200 мм); критерий c-оптимальности (§0).

**Допущения.** Чтобы перевести дизайн из модельного в расчёт на реальных параметрах, оптимизация ведётся по c-оптимальности (минимум дисперсии ρ₂) в допустимом диапазоне размеров. Верхняя граница L_max пока задана заглушкой 200 мм; её строгое значение (плоскостность) требует расчёта по томографии.

In [ ]:
# @title §8.1 Рабочая точка из данных, ограничения, c-оптимальность
# рабочая точка <- params/static.json (09) + params/ct.json (10); ρ2 предпочтительно из КТ.
import os as _os, json as _json
def _ld(p):
    pth = _os.path.join("params", p)
    return _json.load(open(pth, encoding="utf-8")) if _os.path.exists(pth) else {}
_st, _ct = _ld("static.json"), _ld("ct.json")
# Масштаб невязки по каждому испытуемому, Ом: среднеквадратичный остаток совместной
# подгонки при фиксированной толщине (09 §3). Не приборный шум — см. пояснение к §7.
SIGMA_BY_SUBJECT = {"Ник": 7.40, "Георгий": 15.10}
SIGMA_DATA = SIGMA_FIT     # запасное значение, если испытуемый не опознан по имени
if _st:
    OP = {"%s (h=%.0f)" % (n, _st[n]["h"]*1000):
          dict(rho1=_st[n]["rho1"], rho2=(_ct.get(n, {}).get("rho2_ct") or _st[n]["rho2_in"]), h=_st[n]["h"],
               sigma=SIGMA_BY_SUBJECT.get(n, SIGMA_FIT))
          for n in _st}
else:
    OP = {"Ник (h=15)": dict(rho1=4.73, rho2=17.7, h=0.015, sigma=SIGMA_BY_SUBJECT["Ник"]),
          "Георгий (h=40)": dict(rho1=13.2, rho2=20.0, h=0.040, sigma=SIGMA_BY_SUBJECT["Георгий"])}
L_MAX_MM   = 200.0         # верхняя граница плоскостности из КТ/МКЭ (заглушка)

def std_rho2(L_set_m, op, sigma=None, beta=BETA_OP):
    if sigma is None:           # масштаб невязки берётся по испытуемому
        sigma = op.get("sigma", SIGMA_DATA)
    if len(L_set_m) < 3:        # для 3 параметров (ρ1,h,ρ2) нужно ≥3 базы
        return np.inf
    try:
        return fisher(np.asarray(L_set_m), op["rho1"], op["h"], op["rho2"], sigma, beta)["std_rho2"]
    except np.linalg.LinAlgError:
        return np.inf

cur = np.arange(50, 141, 10)/1000.0
print("c-оптимальность = min std(ρ2). L набора ∈ [40, %.0f мм]: мелкие базы закрепляют ρ1,\n"
      "крупные несут ρ2; верх ограничен плоскостностью Lmax из КТ (§8.3).\n" % L_MAX_MM)
for name, op in OP.items():
    print("  %-16s std(ρ2) текущего набора 50–140 = %.3f Ом·м" % (name, std_rho2(cur, op)))

**Анализ результатов.** Для текущего набора 50–140 мм при фактическом масштабе невязки по каждому испытуемому (7.40 Ом у Ника, 15.10 Ом у Георгия) стандартное отклонение оценки ρ₂ равно 112.3 Ом·м у Георгия (h = 40 мм) и 14.07 Ом·м у Ника (h = 15 мм).

Эти значения количественно воспроизводят результат обратной задачи, полученный независимо. У Ника ρ₂ восстановлена как 17.12 Ом·м при стандартном отклонении 14.07 Ом·м, то есть определена, но с неопределённостью порядка самой величины — что соответствует характеристике «оценка шаткая» в [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3. У Георгия стандартное отклонение 112.3 Ом·м при ожидаемом ρ₂ около 18.6 Ом·м превышает величину в шесть раз, что соответствует выходу оценки на верхний предел допустимого диапазона в [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3.1. Совпадение предсказания матрицы Фишера с наблюдаемым поведением обратной задачи подтверждает, что причина неопределимости — обусловленность, а не дефект решателя.

**Результаты и умозаключения.** Установлены рабочая точка и ограничения оптимизации; при большой толщине текущий набор недостаточен, что мотивирует расширение диапазона размеров (§8.3). Поскольку принятое σ включает воспроизводимую систематику отдельных сборок ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §2.2), приведённые стандартные отклонения являются оценками сверху для достижимой неопределённости: калибровка сборок способна снизить σ примерно втрое и во столько же — стандартное отклонение оценки ρ₂.

**Входные данные.** Эластичности как функция формы β = b/a при фиксированном размере сборки (модельный расчёт; все измерения сделаны при β = 0.5).

**Допущения.** Чтобы оценить влияние формы, стандартное отклонение оценки ρ₂ вычисляется в зависимости от β. Так как измерения выполнены только при β = 0.5, результат по β — модельная рекомендация, а не вывод из данных.

In [ ]:
# @title §8.2 Оптимизация формы β = b/a (МОДЕЛЬНО)
betas = np.linspace(0.2, 0.9, 36)
plt.figure(figsize=(8, 5))
for name, op in OP.items():
    s = [std_rho2(cur, op, beta=bb) for bb in betas]
    b_star = betas[int(np.argmin(s))]
    plt.plot(betas, s, label="%s: β*=%.2f" % (name, b_star))
    plt.axvline(b_star, ls=":", lw=0.8)
plt.axvline(0.5, color="k", lw=1.2, label="текущее β=0.5")
plt.xlabel("β = b/a"); plt.ylabel("std($\\hat\\rho_2$), Ом·м"); plt.yscale("log")
plt.title("Форма решётки: c-оптимальность по β (набор 50–140)\n⚠ модельная экстраполяция — измерения только при β=0.5")
plt.legend(); plt.grid(True, which="both"); plt.tight_layout(); plt.show()

**Анализ результатов.** Оптимум по форме лежит внутри интервала: малое β повышает сигнал, но снижает измеряемое напряжение и меняет глубинную чувствительность.

**Результаты и умозаключения.** Рекомендация по β — для будущих сборок; подтверждение требует измерений при других β.

**Входные данные.** Калиброванная рабочая точка, ограничения на размер сборки и критерий c-оптимальности; оптимизация ведётся по набору из десяти размеров в диапазоне 40–200 мм.

**Допущения.** Чтобы найти оптимальный набор размеров под каждую толщину, минимизируется стандартное отклонение оценки ρ₂ при ограничениях на диапазон размеров.

In [ ]:
# @title §8.3 Оптимизация набора {L_k} под ограничениями + робастность по h
# Нижняя граница набора — физический минимум базы (НЕ Lmin): мелкие сборки нужны,
# чтобы закрепить ρ1,h и разорвать вырождение h<->ρ2 (§2, §8-9 дизайн-доку).
# Верхняя — Lmax(h) из КТ (плоскостность).
L_MIN_MM = 40.0
GRID = np.arange(L_MIN_MM, int(L_MAX_MM)+1, 10)/1000.0      # сетка возможных размеров, м
K = 10                                                       # число решёток в наборе

def greedy_design(op, K):
    cand = list(GRID)
    chosen = [cand[0], cand[len(cand)//2], cand[-1]]          # стартовый спред для невырожденной M
    while len(chosen) < min(K, len(cand)):
        rest = [L for L in cand if L not in chosen]
        L_best = min(rest, key=lambda L: std_rho2(np.array(sorted(chosen+[L])), op))
        chosen.append(L_best)
    return np.array(sorted(chosen))

print("%-16s | std тек.50–140 | оптим. набор {L_k}, мм                 | std опт." % "субъект")
print("-"*92)
for name, op in OP.items():
    opt = greedy_design(op, K)
    print("%-16s |    %.3f      | %-38s | %.3f"
          % (name, std_rho2(cur, op), str([int(x*1000) for x in opt]), std_rho2(opt, op)))

# робастность: средняя std(ρ2) по диапазону h для текущего и оптимизированного наборов
plt.figure(figsize=(8, 5))
for name, op in OP.items():
    opt = greedy_design(op, K)
    cur_h = [std_rho2(cur, dict(op, h=h)) for h in hs]
    opt_h = [std_rho2(opt, dict(op, h=h)) for h in hs]
    ln, = plt.plot(hs*1000, cur_h, "--", label="%s: 50–140" % name)
    plt.plot(hs*1000, opt_h, "-", color=ln.get_color(), label="%s: оптим." % name)
plt.xlabel("толщина h, мм"); plt.ylabel("std($\\hat\\rho_2$), Ом·м"); plt.yscale("log")
plt.title("Робастность набора по толщине: текущий vs оптимизированный")
plt.legend(fontsize=8); plt.grid(True, which="both"); plt.tight_layout(); plt.show()

**Анализ результатов.** Оптимальный набор смещён к крупным размерам и снижает стандартное отклонение оценки ρ₂: у Георгия со 112.3 до 34.6 Ом·м, у Ника с 14.07 до 5.67 Ом·м, то есть примерно втрое у обоих. Отношение не зависит от принятого масштаба невязки, поскольку стандартное отклонение пропорционально σ.

Даже после оптимизации набора у Георгия стандартное отклонение (34.6 Ом·м) остаётся вдвое больше самой оцениваемой величины (около 18.6 Ом·м). Следовательно, при толщине мягких тканей 40 мм одним лишь выбором размеров сборок задача не решается: требуется либо снижение расхождения модели с данными, либо иная измерительная схема.

**Результаты и умозаключения.** Расширение набора вверх (до границы плоскостности) втрое снижает неопределённость ρ₂; конкретная верхняя граница требует расчёта плоскостности по томографии (открытый вопрос, [06](34.01_Контракт_проверки_двуслойной_модели.md) §2.5). Для толстых мягких тканей расширения набора недостаточно, и первоочередным становится снижение σ: устранение систематики отдельных сборок калибровкой ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §2.2.1) даёт выигрыш того же порядка, что и оптимизация геометрии, и не требует новых сборок.

### §8.4. Выводы

- При фиксированном $\beta=0.5$ расширение набора вверх (к $L_{\max}$ из КТ) снижает $\mathrm{std}(\hat\rho_2)$, особенно для толстых тканей — крупные базы несут $\rho_2$ (ср. §6).
- Оптимум по $\beta$ — **модельное предсказание** для будущих сборок; подтверждается только измерениями при других $\beta$.
- Ключевое ограничение — $L_{\max}(h,\text{анатомия})$: без него оптимизатор бесконтрольно растит $L$. Замените заглушку `L_MAX_MM` границей плоскостности из КТ/МКЭ (§2.5, §3.3 документа перекрёстной проверки).
- Рабочую точку $(\rho_1,\rho_2,h)$ и $\sigma_Z$ обновляйте из `09_Статическая_оценка_параметров.ipynb`.